Run sequence:
- Train model on id train.pt
- Run fit_mahalanobis on id train_loader
- Get results for id data with id test_loader  
- Get results for ood data with ood test_loader  (rename files!)

In [1]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import numpy as np

from exp.exp_forecasting import Exp_Forecasting
from exp.exp_classification import Exp_Classification
from utils.tools import (
    set_seed,
    print_formatted_dict,
)
from dataset_loader.dataset_loader import update_args_from_dataset
from main import trainable, get_args_from_parser, update_args

/opt/anaconda3/envs/timedrl/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_dataset = torch.load("./dataset/classification/HAR/train.pt")
val_dataset = torch.load("./dataset/classification/HAR/val.pt")
test_dataset = torch.load("./dataset/classification/HAR/test.pt")

In [3]:
labels = train_dataset.get("labels").numpy()

unique, counts = np.unique(labels, return_counts=True)

In [4]:
print(dict(zip(unique, counts)))

{0.0: 979, 2.0: 780, 3.0: 1024, 5.0: 1127}


# Train model

In [11]:
# WISDM

task_name = "classification"

data_name = "WISDM"  # (256, 3)

num_workers = 4  # It doesn't really matter since we don't use it in dataloader
batch_size = 16

train_together = False  # default
get_i = "cls"
encoder_arch = "transformer_encoder"  # default
data_aug = "none"
disable_stop_gradient = False
disable_predictive_loss = False  # default
disable_contrastive_loss = False  # default
pretrain_data_percent = 100  # default
linear_eval_data_percent = 100  # default
disable_freeze_encoder = False  # default
set_seed(seed=2023)

# Setup args
args = get_args_from_parser()
args.overwrite_args = True

# Setup fixed params
fixed_params = {
    "task_name": task_name,
    "data_name": data_name,
    "num_workers": num_workers,
    "batch_size": batch_size,
    "train_together": train_together,
    "get_i": get_i,
    "encoder_arch": encoder_arch,
    "data_aug": data_aug,
    "disable_stop_gradient": disable_stop_gradient,
    "disable_predictive_loss": disable_predictive_loss,
    "disable_contrastive_loss": disable_contrastive_loss,
    "pretrain_data_percent": pretrain_data_percent,
    "linear_eval_data_percent": linear_eval_data_percent,
    "disable_freeze_encoder": disable_freeze_encoder,
}

# Setup tunable params
# TODO: copy `config` from `exp_settings_and_results` (be careful with the boolean values)
tunable_params = {
    "pretrain_optim": "AdamW",
    "pretrain_learning_rate": 0.0001407743941654141,
    "pretrain_lradj": "constant",
    "pretrain_weight_decay": 0.00008709261322555189,
    "pretrain_epochs": 10, # 30 - best
    "contrastive_weight": 0.1,
    "linear_eval_optim": "AdamW",
    "linear_eval_learning_rate": 0.0014077439416541409,
    "linear_eval_lradj": "constant",
    "linear_eval_weight_decay": 0.00017514842046704846,
    "linear_eval_epochs": 7, # 30 - best
    "patience": 5,
    # "pos_embed_type": "learnable",
    # "token_embed_type": "conv",
    # "token_embed_kernel_size": 3,
    # "dropout": 0.2,
    # "base_d_model": 64,
    # "n_layers": 1,
    # "n_heads": 2,
    # "patch_len": 16,
    # "stride": 16,
    # "enable_channel_independence": True,
    # "seq_len": 128,
    "pos_embed_type": "learnable",
    "token_embed_type": "conv",
    "token_embed_kernel_size": 5,
    "dropout": 0.2,
    "base_d_model": 128,
    "n_layers": 3,
    "n_heads": 2,
    "patch_len": 4,
    "stride": 4,
    "enable_channel_independence": False,
    "seq_len": 168
}

In [5]:
# HAR

task_name = "classification"

data_name = "HAR"  # (256, 3)

num_workers = 4  # It doesn't really matter since we don't use it in dataloader
batch_size = 16

train_together = False  # default
get_i = "cls"
encoder_arch = "transformer_encoder"  # default
data_aug = "none"
disable_stop_gradient = False
disable_predictive_loss = False  # default
disable_contrastive_loss = False  # default
pretrain_data_percent = 100  # default
linear_eval_data_percent = 100  # default
disable_freeze_encoder = False  # default
set_seed(seed=2023)

# Setup args
args = get_args_from_parser()
args.overwrite_args = True

# Setup fixed params
fixed_params = {
    "task_name": task_name,
    "data_name": data_name,
    "num_workers": num_workers,
    "batch_size": batch_size,
    "train_together": train_together,
    "get_i": get_i,
    "encoder_arch": encoder_arch,
    "data_aug": data_aug,
    "disable_stop_gradient": disable_stop_gradient,
    "disable_predictive_loss": disable_predictive_loss,
    "disable_contrastive_loss": disable_contrastive_loss,
    "pretrain_data_percent": pretrain_data_percent,
    "linear_eval_data_percent": linear_eval_data_percent,
    "disable_freeze_encoder": disable_freeze_encoder,
}

# Setup tunable params
# TODO: copy `config` from `exp_settings_and_results` (be careful with the boolean values)
tunable_params = {
    "pretrain_optim": "AdamW",
    "pretrain_learning_rate": 0.00015687380445185992,
    "pretrain_lradj": "warmup",
    "pretrain_weight_decay": 0.004848209729531062,
    "pretrain_epochs": 10, # 30 - best
    "contrastive_weight": 0.1,
    "linear_eval_optim": "AdamW",
    "linear_eval_learning_rate": 0.0015687380445185992,
    "linear_eval_lradj": "type3",
    "linear_eval_weight_decay": 0.0017207231191582646,
    "linear_eval_epochs": 7, # 30 - best
    "patience": 5,
    "pos_embed_type": "fixed",
    "token_embed_type": "conv",
    "token_embed_kernel_size": 3,
    "dropout": 0.2,
    "base_d_model": 128,
    "n_layers": 3,
    "n_heads": 2,
    "patch_len": 8,
    "stride": 4,
    "enable_channel_independence": False,
    "seq_len": 512
}

In [ ]:
return_metrics = trainable(tunable_params, fixed_params, args)
print_formatted_dict(return_metrics)

In [6]:
# 1. Setup and Update Args
args = update_args(args, fixed_params, tunable_params)

# 2. Initialize the Experiment
exp = Exp_Classification(args)

# 3. Run Training (this trains both encoder and linear head)
if args.train_together:
    train_metrics = exp.train_together(use_tqdm=True)
else:
    train_metrics = exp.train(use_tqdm=True)

### [Fixed] Set task_name to classification
### [Fixed] Set data_name to HAR
### [Fixed] Set num_workers to 4
### [Fixed] Set batch_size to 16
### [Fixed] Set train_together to False
### [Fixed] Set get_i to cls
### [Fixed] Set encoder_arch to transformer_encoder
### [Fixed] Set data_aug to none
### [Fixed] Set disable_stop_gradient to False
### [Fixed] Set disable_predictive_loss to False
### [Fixed] Set disable_contrastive_loss to False
### [Fixed] Set pretrain_data_percent to 100
### [Fixed] Set linear_eval_data_percent to 100
### [Fixed] Set disable_freeze_encoder to False
### [Tunable] Set pretrain_optim to AdamW
### [Tunable] Set pretrain_learning_rate to 0.00015687380445185992
### [Tunable] Set pretrain_lradj to warmup
### [Tunable] Set pretrain_weight_decay to 0.004848209729531062
### [Tunable] Set pretrain_epochs to 10
### [Tunable] Set contrastive_weight to 0.1
### [Tunable] Set linear_eval_optim to AdamW
### [Tunable] Set linear_eval_learning_rate to 0.0015687380445185992
##

Class Distribution Across Datasets:

┏━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Class ┃         Train ┃   Validation ┃         Test ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│   0.0 │  979 (25.04%) │ 247 (24.82%) │ 496 (25.51%) │
│   2.0 │  780 (19.95%) │ 206 (20.70%) │ 420 (21.60%) │
│   3.0 │ 1024 (26.19%) │ 262 (26.33%) │ 491 (25.26%) │
│   5.0 │ 1127 (28.82%) │ 280 (28.14%) │ 537 (27.62%) │
└───────┴───────────────┴──────────────┴──────────────┘

----------------------------------------
### HAR ###
N: 6849 (train: 3910, val: 995, test: 1944)
C: 9
K: 4
T: 128


Class Distribution Across Datasets:

...

100%|██████████| 63/63 [00:29<00:00,  2.17it/s]


>>>>> Calculate testing metrics >>>>>


100%|██████████| 122/122 [00:56<00:00,  2.16it/s]


┏━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Epoch ┃ Valid LOSS ┃ Valid ACC ┃ Valid MF1 ┃ Valid KAPPA ┃ Test LOSS ┃ Test ACC ┃ Test MF1 ┃ Test KAPPA ┃
┡━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━┩
│ 1     │ 0.010      │ 0.952     │ 0.954     │ 0.935       │ 0.024     │ 0.899    │ 0.899    │ 0.864      │
└───────┴────────────┴───────────┴───────────┴─────────────┴───────────┴──────────┴──────────┴────────────┘

EarlyStopping counter: 3 out of 5 (best val_loss: 0.009679, current val_loss: 0.010147)
Early stopping
Updating learning rate to 9.263241279077878e-05
------------------------------------------------------------------
best_pretrain_loss: 0.083
predictive_loss: 0.169
contrastive_loss: -0.860
best_test_acc: 0.899
best_test_mf1: 0.899
best_test_kappa: 0.864
best_pretrain_epoch: 10
best_best_test_acc_epoch: 10


In [7]:
# WITH ID DATA!!!!!!

train_loader, val_loader, test_loader = exp._get_data()

exp.fit_mahalanobis(train_loader)

----------------------------------------
### HAR ###
N: 6849 (train: 3910, val: 995, test: 1944)
C: 9
K: 4
T: 128


Class Distribution Across Datasets:

┏━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Class ┃         Train ┃   Validation ┃         Test ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│   0.0 │  979 (25.04%) │ 247 (24.82%) │ 496 (25.51%) │
│   2.0 │  780 (19.95%) │ 206 (20.70%) │ 420 (21.60%) │
│   3.0 │ 1024 (26.19%) │ 262 (26.33%) │ 491 (25.26%) │
│   5.0 │ 1127 (28.82%) │ 280 (28.14%) │ 537 (27.62%) │
└───────┴───────────────┴──────────────┴──────────────┘

>>>>> Fitting Mahalanobis Parameters (Train ID Data) >>>>>


Extracting Train Latents: 100%|██████████| 245/245 [01:52<00:00,  2.18it/s]


Mahalanobis fitted. 95% TPR Threshold: 27.1165


In [10]:
# 4. GET THE DATA LOADERS
# We need the loaders to pass into your new analysis function
train_loader, val_loader, test_loader = exp._get_data()

# 5. CALL THE ANALYSIS FUNCTION
print(">>>>> Running Detailed Analysis for OOD/Confusion Matrix >>>>>")
analysis_results = exp.get_detailed_analysis_dict(test_loader, T=1000, noise=0.001)

# Now you have everything!
print(f"Latents shape: {analysis_results['latents'].shape}")
print(f"Logits shape: {analysis_results['logits'].shape}")

# Example: Accessing targets and preds for a custom confusion matrix
from sklearn.metrics import confusion_matrix
y_true = analysis_results['targets']
y_pred = analysis_results['preds']
print(confusion_matrix(y_true, y_pred))

----------------------------------------
### HAR ###
N: 3450 (train: 1971, val: 476, test: 1003)
C: 9
K: 2
T: 128


Class Distribution Across Datasets:

┏━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Class ┃         Train ┃   Validation ┃         Test ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│   1.0 │  873 (44.29%) │ 200 (42.02%) │ 471 (46.96%) │
│   4.0 │ 1098 (55.71%) │ 276 (57.98%) │ 532 (53.04%) │
└───────┴───────────────┴──────────────┴──────────────┘

>>>>> Running Detailed Analysis for OOD/Confusion Matrix >>>>>


Evaluating OOD Metrics: 100%|██████████| 63/63 [01:10<00:00,  1.13s/it]

Latents shape: (1003, 256)
Logits shape: (1003, 6)
[[  0   0   0   0   0   0]
 [367   0   2 102   0   0]
 [  0   0   0   0   0   0]
 [  0   0   0   0   0   0]
 [ 14   0   1 396   0 121]
 [  0   0   0   0   0   0]]


In [11]:
# To Save
file_path = "HAR_n3_h2_ood_v2.npz"
np.savez(file_path, **analysis_results)
print(f"Saved analysis results to {file_path}")

Saved analysis results to HAR_n3_h2_ood_v2.npz


In [18]:
def calculate_simple_auroc(id_scores, ood_scores, direction='higher_is_id'):
    """
    Calculates AUROC using the Wilcoxon-Mann-Whitney statistic.
    """
    # Ensure they are numpy arrays
    id_s = np.asarray(id_scores)
    ood_s = np.asarray(ood_scores)

    # If Mahalanobis, we flip so that higher values = ID
    if direction == 'lower_is_id':
        id_s = -id_s
        ood_s = -ood_s

    # Combine scores and create labels (1 for ID, 0 for OOD)
    all_scores = np.concatenate([id_s, ood_s])
    labels = np.concatenate([np.ones(len(id_s)), np.zeros(len(ood_s))])
    
    # Sort by score to calculate ranks
    indices = np.argsort(all_scores)
    sorted_labels = labels[indices]
    
    ranks = np.arange(1, len(all_scores) + 1)
    pos_ranks = ranks[sorted_labels == 1]
    
    n_pos = len(id_s)
    n_neg = len(ood_s)
    
    # Mann-Whitney U Logic
    u_stat = np.sum(pos_ranks) - (n_pos * (n_pos + 1) / 2)
    return u_stat / (n_pos * n_neg)


def tune_odin_parameters(exp, valid_loader, ood_valid_loader):
    """
    Finds the best T and noise for ODIN using Validation data.
    """
    best_auroc = 0
    best_params = {'T': 1000, 'noise': 0.001}
    
    # Standard research ranges for ODIN
    T_list = [1, 10, 100, 1000]
    noise_list = [0, 0.0001, 0.0005, 0.001, 0.002, 0.004]

    print(">>>>> Starting ODIN Grid Search >>>>>")
    for T in T_list:
        for noise in noise_list:
            # 1. Get ODIN scores for ID and OOD validation sets
            id_val = exp.get_detailed_analysis_dict(valid_loader, T=T, noise=noise)
            ood_val = exp.get_detailed_analysis_dict(ood_valid_loader, T=T, noise=noise)
            
            # 2. Calculate AUROC for this specific T/noise combo
            # (Using a simple AUROC helper)
            current_auroc = calculate_simple_auroc(id_val['odin_score'], ood_val['odin_score'])
            
            print(f"T: {T:4d} | Noise: {noise:.4f} | AUROC: {current_auroc:.4f}")
            
            if current_auroc > best_auroc:
                best_auroc = current_auroc
                best_params = {'T': T, 'noise': noise}
                
    print(f"\nBest ODIN Params: {best_params} with AUROC: {best_auroc:.4f}")
    return best_params

In [19]:
train_loader, val_loader, id_test_loader = exp._get_data()

----------------------------------------
### WISDM ###
N: 2362 (train: 1504, val: 380, test: 478)
C: 3
K: 4
T: 256


Class Distribution Across Datasets:

┏━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Class ┃         Train ┃   Validation ┃         Test ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│     0 │ 1027 (68.28%) │ 263 (69.21%) │ 339 (70.92%) │
│     2 │  152 (10.11%) │   31 (8.16%) │   38 (7.95%) │
│     3 │   122 (8.11%) │   35 (9.21%) │   23 (4.81%) │
│     5 │  203 (13.50%) │  51 (13.42%) │  78 (16.32%) │
└───────┴───────────────┴──────────────┴──────────────┘

In [20]:
train_loader, val_loader, ood_test_loader = exp._get_data()

----------------------------------------
### WISDM ###
N: 1729 (train: 1113, val: 275, test: 341)
C: 3
K: 2
T: 256


Class Distribution Across Datasets:

┏━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Class ┃        Train ┃   Validation ┃         Test ┃
┡━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│     1 │ 830 (74.57%) │ 225 (81.82%) │ 258 (75.66%) │
│     4 │ 283 (25.43%) │  50 (18.18%) │  83 (24.34%) │
└───────┴──────────────┴──────────────┴──────────────┘

In [21]:
tune_odin_parameters(exp, id_test_loader, ood_test_loader)

>>>>> Starting ODIN Grid Search >>>>>


Evaluating OOD Metrics: 100%|██████████| 22/22 [00:41<00:00,  1.89s/it]


T:    1 | Noise: 0.0000 | AUROC: 0.8139


Evaluating OOD Metrics: 100%|██████████| 22/22 [00:43<00:00,  1.99s/it]


T:    1 | Noise: 0.0001 | AUROC: 0.8087


...

T:   10 | Noise: 0.0001 | AUROC: 0.7273


Evaluating OOD Metrics: 100%|██████████| 22/22 [00:49<00:00,  2.24s/it]


T:   10 | Noise: 0.0005 | AUROC: 0.7160


Evaluating OOD Metrics:  43%|████▎     | 13/30 [00:32<00:41,  2.47s/it]


KeyboardInterrupt: 

# Notes and code snippets for later

In [8]:
# Key,Source in TimeDRL,Purpose in OOD Detection
# latents,Output of the Encoder (i1​),"These are the ""raw"" feature vectors. In OOD detection, you use these to see if OOD samples cluster in a different area of the latent space than the training classes."
# logits,Output of the Linear Head,"The raw scores before normalization. These are essential for Energy-based scores, which are often more reliable for OOD than probabilities."
# probs,softmax(logits),"The probability distribution across all known classes. For OOD data, you expect this to be ""flatter"" (higher entropy)."
# preds,argmax(probs),"The final class assignment. For OOD data, the model is forced to choose a class, which is technically a ""false positive."""
# targets,Original Ground Truth,Used to calculate the Confusion Matrix and to identify which samples were your removed OOD classes.
# max_conf,max(probs),"The Maximum Softmax Probability (MSP). This is your primary baseline score: high for known classes, low for OOD."

so i should train the model using only the 4 class train data and validate it with a test and val dataset where also only those 4 classes exist, then after the training i should run this get_detailed_analysis_dict function with 2 cases: the first where i should only give it those 4 classes that the model knows and second only those 2 classes that it doesnt know?

In [ ]:
def get_detailed_analysis_dict(self, data_loader):
    self.model.eval()
    self.linear_eval.eval()
    
    raw_latents, raw_logits, raw_targets = [], [], []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.float().to(self.device)
            # Taking i_1 as the primary representation
            _, _, _, _, i_1, _, _, _ = self.model(batch_x)
            logits = self.linear_eval(i_1)
            
            raw_latents.append(i_1.detach().cpu().numpy())
            raw_logits.append(logits.detach().cpu().numpy())
            raw_targets.append(batch_y.numpy())

    latents = np.concatenate(raw_latents, axis=0)
    logits = np.concatenate(raw_logits, axis=0)
    targets = np.concatenate(raw_targets, axis=0)
    min_len = min(len(latents), len(targets))
    
    latents, logits, targets = latents[:min_len], logits[:min_len], targets[:min_len]

    # --- MAHALANOBIS CALCULATION ---
    # Note: You should pre-calculate 'self.class_means' and 'self.precision_matrix' 
    # using your TRAIN set before calling this on TEST/OOD data.
    
    mahal_distances = []
    if hasattr(self, 'class_means') and hasattr(self, 'precision_matrix'):
        for z in latents:
            # Calculate distance to each of the 4 class centers
            dists = []
            for m in self.class_means:
                diff = z - m
                # d^2 = diff @ Precision @ diff.T
                d2 = np.dot(np.dot(diff, self.precision_matrix), diff.T)
                dists.append(np.sqrt(d2))
            # The score is the distance to the CLOSEST known class
            mahal_distances.append(min(dists))
    else:
        mahal_distances = np.zeros(min_len) # Fallback if not fitted
    # -------------------------------

    probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
    
    return {
        "latents": latents,
        "logits": logits,
        "targets": targets,
        "preds": np.argmax(probs, axis=1),
        "max_conf": np.max(probs, axis=1),
        "mahal_distance": np.array(mahal_distances)
    }

In [ ]:
def fit_mahalanobis(self, train_loader):
    self.model.eval()
    all_latents = []
    all_labels = []
    
    with torch.no_grad():
        for batch_x, batch_y in train_loader:
            _, _, _, _, i_1, _, _, _ = self.model(batch_x.float().to(self.device))
            all_latents.append(i_1.cpu().numpy())
            all_labels.append(batch_y.numpy())
            
    z = np.concatenate(all_latents)
    y = np.concatenate(all_labels)
    
    # 1. Calculate Mean for each class (0, 1, 4, 5)
    self.class_means = [np.mean(z[y == c], axis=0) for c in np.unique(y)]
    
    # 2. Calculate Shared Covariance Matrix
    centered_z = np.zeros_like(z)
    for i, c in enumerate(np.unique(y)):
        centered_z[y == c] = z[y == c] - self.class_means[i]
    
    cov = np.cov(centered_z.T)
    # Use Pseudo-Inverse (pinv) to handle potentially singular matrices
    self.precision_matrix = np.linalg.pinv(cov) 
    print("Mahalanobis parameters fitted.")

In [ ]:
That is a brilliant plan. You are moving from Probability-based detection (which is easily fooled by Softmax overconfidence) to Distance-based detection (which looks at the actual physical "identity" of the data in the latent space).Since you are using Exp_classification, you should implement this in two stages:Calculate the Statistics (Mean and Covariance) during a "fitting" phase using your training data.Calculate the Distance for every sample inside your get_detailed_analysis_dict.The Mahalanobis FormulaThe Mahalanobis distance is better than standard Euclidean distance because it accounts for how "spread out" each class is.$$d_M(x, \mu) = \sqrt{(x - \mu)^T \Sigma^{-1} (x - \mu)}$$Updated get_detailed_analysis_dictTo make this work, your class needs to "remember" the centers of the 4 known classes. I've added the Mahalanobis calculation directly into your function logic:Pythondef get_detailed_analysis_dict(self, data_loader):
    self.model.eval()
    self.linear_eval.eval()
    
    raw_latents, raw_logits, raw_targets = [], [], []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.float().to(self.device)
            # Taking i_1 as the primary representation
            _, _, _, _, i_1, _, _, _ = self.model(batch_x)
            logits = self.linear_eval(i_1)
            
            raw_latents.append(i_1.detach().cpu().numpy())
            raw_logits.append(logits.detach().cpu().numpy())
            raw_targets.append(batch_y.numpy())

    latents = np.concatenate(raw_latents, axis=0)
    logits = np.concatenate(raw_logits, axis=0)
    targets = np.concatenate(raw_targets, axis=0)
    min_len = min(len(latents), len(targets))
    
    latents, logits, targets = latents[:min_len], logits[:min_len], targets[:min_len]

    # --- MAHALANOBIS CALCULATION ---
    # Note: You should pre-calculate 'self.class_means' and 'self.precision_matrix' 
    # using your TRAIN set before calling this on TEST/OOD data.
    
    mahal_distances = []
    if hasattr(self, 'class_means') and hasattr(self, 'precision_matrix'):
        for z in latents:
            # Calculate distance to each of the 4 class centers
            dists = []
            for m in self.class_means:
                diff = z - m
                # d^2 = diff @ Precision @ diff.T
                d2 = np.dot(np.dot(diff, self.precision_matrix), diff.T)
                dists.append(np.sqrt(d2))
            # The score is the distance to the CLOSEST known class
            mahal_distances.append(min(dists))
    else:
        mahal_distances = np.zeros(min_len) # Fallback if not fitted
    # -------------------------------

    probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
    
    return {
        "latents": latents,
        "logits": logits,
        "targets": targets,
        "preds": np.argmax(probs, axis=1),
        "max_conf": np.max(probs, axis=1),
        "mahal_distance": np.array(mahal_distances)
    }
Important: How to "Fit" the distance parametersBefore running the analysis on OOD data, you need to "teach" the model what the 4 classes look like. Add this method to your Exp_classification and run it once using your ID Train Loader:Pythondef fit_mahalanobis(self, train_loader):
    self.model.eval()
    all_latents = []
    all_labels = []
    
    with torch.no_grad():
        for batch_x, batch_y in train_loader:
            _, _, _, _, i_1, _, _, _ = self.model(batch_x.float().to(self.device))
            all_latents.append(i_1.cpu().numpy())
            all_labels.append(batch_y.numpy())
            
    z = np.concatenate(all_latents)
    y = np.concatenate(all_labels)
    
    # 1. Calculate Mean for each class (0, 1, 4, 5)
    self.class_means = [np.mean(z[y == c], axis=0) for c in np.unique(y)]
    
    # 2. Calculate Shared Covariance Matrix
    centered_z = np.zeros_like(z)
    for i, c in enumerate(np.unique(y)):
        centered_z[y == c] = z[y == c] - self.class_means[i]
    
    cov = np.cov(centered_z.T)
    # Use Pseudo-Inverse (pinv) to handle potentially singular matrices
    self.precision_matrix = np.linalg.pinv(cov) 
    print("Mahalanobis parameters fitted.")
Why this solves your 99% confidence problem:In your previous plot, the OOD samples had high softmax scores because they were "on the correct side" of the decision boundary, but they were likely very far away from the actual cluster centers.Softmax only cares about the angle of the vector.Mahalanobis cares about the position.An OOD sample might have the same "angle" as Class 1 (giving it 99% confidence), but its Mahalanobis distance will be huge because it is physically located in a "dead zone" of the latent space where no training data ever existed.Would you like me to show you how to visualize these distances as a "Heatmap" over a 2D t-SNE plot so you can see the "Safety Zones" around your classes?